# Chạy thử đầu-cuối T3 → T8 trên một kho nhỏ, trong một notebook

Chạy cả đường ống trong một phiên để kiểm tra và đo thời gian.
- `CORPUS = 'sample'`: kho mẫu (1.924 bài → 4.851 đoạn). Chỉ để kiểm tra, **không** dùng để nộp bài hay gán nhãn:
  kho mẫu là 2.000 bài ngẫu nhiên, gần như không liên quan tới câu hỏi.
- `CORPUS = 'full'`: dùng `corpus_clean/` và `chunks/structure/` đang có trong `mir-data`, ví dụ phần kho đã crawl
  được. Cho run thật (Dense, LSR, BM25, G, G sau xếp hạng lại) để gán nhãn thử (`docs/t7_danh_gia.md`) trước khi có
  toàn kho. Thời gian T3 tỷ lệ với số đoạn: khoảng 185 đoạn/s trên T4 ×2.

**Cài đặt notebook:** Accelerator = **GPU T4 ×2**, Internet = **On**.
**Add Input:** `mir-code`, `mir-data`. Notebook tự tìm đường dẫn.

`FULL_QUERIES = False`: 50 câu hỏi mẫu; với kho mẫu cả notebook khoảng 5 phút. `True`: đủ 1.200 câu, bài nộp đúng định
dạng (riêng T6 khoảng 28 phút).

Các bước: (1) chuẩn bị → (2) đối chiếu với FlagEmbedding → (3) đo tốc độ → (4) T3 → (5) T4 → (6) T5 → (7) T6
→ (8) T8 → (9) tổng kết. Số đo lần chạy 05/10/2026 (kho mẫu) ghi ở `docs/NOTES.md`.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây; phần còn lại giống nhau ở mọi notebook.
NB_VERSION = 'e2e-v4 (05/10/2026)'
CORPUS = 'sample'           # 'sample': kho mẫu; 'full': corpus_clean/ + chunks/ đang có trong mir-data
FULL_QUERIES = False        # True: 1.200 câu hỏi thật; False: 50 câu hỏi của tập mẫu (nhanh)
OVERRIDES = {'encode.shard_size': 2500} if CORPUS == 'sample' else {}   # kho mẫu: khối nhỏ để 2 GPU chia việc
if not FULL_QUERIES:
    OVERRIDES['retrieve.queries'] = 'sample/query.parquet'

import glob, os, shutil, sys
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')          # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mir/__init__.py', recursive=True), key=len)
assert hits, 'Chưa thêm dataset mir-code (Add Input → Your Datasets)'
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)                     # luôn chép lại bản mir-code mới nhất
shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mir.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, OVERRIDES, corpus=CORPUS)
import json

In [ ]:
# (2) Đối chiếu mir.encode với FlagEmbedding (thư viện chính thức) trên 32 đoạn.
#     Mốc 05/10/2026: dense cos nhỏ nhất 0,99972; sparse cùng tập token 32/32; chênh trọng số 0,0000
FLAG = f'{KAGGLE}/tmp/flag'                 # cài vào thư mục riêng để không đổi phiên bản transformers của môi trường
sh(f'python -m pip install -q --no-deps --target {FLAG} FlagEmbedding')
sys.path.insert(0, FLAG)
import numpy as np, pyarrow.parquet as pq, torch
from mir import encode
ch = pq.read_table(f'{K.data}/sample/chunks/structure').to_pydict()
titles = encode.load_titles(f'{K.data}/sample/corpus_clean')
texts = encode.make_inputs({'doc_id': ch['doc_id'][:32], 'text': ch['text'][:32]}, titles)
mine = encode.M3Encoder.load('BAAI/bge-m3', 'cuda:0', fp16=True, max_length=512)
d1, s1, _ = mine.encode(texts)
try:
    from FlagEmbedding import BGEM3FlagModel
    ref = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, devices=['cuda:0'])
    out = ref.encode(texts, max_length=512, return_dense=True, return_sparse=True)
    cos = (d1.astype(np.float32) * out['dense_vecs'].astype(np.float32)).sum(1)
    same = sum(set(i.tolist()) == {int(k) for k in lw} for (i, _), lw in zip(s1, out['lexical_weights']))
    worst = max(abs(float(lw.get(str(t), lw.get(int(t), 0))) - float(x))
                for (ids, w), lw in zip(s1, out['lexical_weights']) for t, x in zip(ids, w))
    print(f'dense cos nhỏ nhất = {cos.min():.5f} | sparse: cùng tập token {same}/32 | chênh trọng số lớn nhất = {worst:.4f}')
    del ref
except Exception as e:
    print('Không đối chiếu được với FlagEmbedding:', type(e).__name__, str(e)[:300])
del mine; torch.cuda.empty_cache()

In [ ]:
# (3) Đo tốc độ mã hoá: 2 tiến trình cùng lúc trên 2 GPU, mỗi tiến trình 2.000 đoạn → ước lượng cho toàn kho
EST = 8_500_000             # số đoạn ước lượng của toàn kho (docs/NOTES.md, T2)
cmd = f'python -m mir.encode --config {K.cfg} --bench 2000 --est-chunks {EST}'
par([f'{cmd} --device cuda:{g}' for g in range(2)], 'bench')
res = [json.load(open(f'{K.work}/results/t3_bench_cuda{g}.json')) for g in range(2)]
rate = sum(r['chunks_per_s'] for r in res)
print(f'TỔNG 2 GPU: {rate:.0f} đoạn/s | toàn kho {EST:,} đoạn ≈ {EST / rate / 3600:.1f} giờ trên 1 máy T4×2 '
      f'(≈ {EST / rate / 3600 / 3:.1f} giờ/người nếu 3 người) | dung lượng ≈ {res[0]["est_storage_gb"]} GB | '
      f'bộ nhớ GPU tối đa {max(r["peak_gpu_mem_gb"] for r in res)} GB')

In [ ]:
# (4) T3 — mã hoá mọi đoạn của kho (2 GPU, mỗi GPU một nửa số khối) → emb/structure/
cmd = f'python -m mir.encode --config {K.cfg}'
par([f'{cmd} --shard {g}/2 --device cuda:{g}' for g in range(2)], 'T3_encode')

In [ ]:
# (5) T4 — chỉ mục BM25 (CPU) và dense (FAISS; kho nhỏ → IndexFlatIP). Nhánh LSR đọc thẳng từ emb ở bước sau.
sh(f'python -m mir.index bm25-tokenize --config {K.cfg}', 'T4_bm25_tokenize')
sh(f'python -m mir.index bm25-build --config {K.cfg}', 'T4_bm25_build')
sh(f'python -m mir.index dense --config {K.cfg}', 'T4_dense')
print(open(f'{K.work}/index/structure/dense.json').read())

In [ ]:
# (6) T5 — mã hoá câu hỏi, đo nlist/nprobe, truy hồi 3 nhánh (độ sâu 1000), gộp RRF A–G + có trọng số
sh(f'python -m mir.retrieve qencode --config {K.cfg} --device cuda:0', 'T5_qencode')
sh(f'python -m mir.index dense-tune --config {K.cfg} --qemb {K.work}/qemb/query.npz', 'T4_dense_tune')
sh(f'python -m mir.retrieve run --config {K.cfg} --branch all', 'T5_retrieve')
sh(f'python -m mir.fuse --config {K.cfg}', 'T5_fuse')
import pandas as pd
for f in sorted(glob.glob(f'{K.work}/runs/*.parquet')):
    d = pd.read_parquet(f)
    print(f'{os.path.basename(f):38s} {len(d):>9,} dòng | {d.qid.nunique():>5} câu hỏi có kết quả')

In [ ]:
# (7) T6 — xếp hạng lại top-100 của cấu hình G (Dense+LSR+BM25) bằng ColBERT trên 2 GPU, rồi gộp 2 phần
RUN = 'structure_G'
cmd = f'python -m mir.rerank --config {K.cfg} --run {K.work}/runs/{RUN}.parquet'
par([f'{cmd} --shard {g}/2 --device cuda:{g}' for g in range(2)], 'T6_rerank')
sh(f'{cmd} --shard 0/2 --device cuda:0')           # đủ 2 phần → gộp thành runs/{RUN}_rr.parquet

In [ ]:
# (8) T8 — bài nộp (kiểm tra đủ câu hỏi, doc_id hợp lệ, chunk_text nguyên văn) → submissions/*.zip
for r in ('structure_G', 'structure_G_rr'):
    sh(f'python -m mir.submit --config {K.cfg} --run {K.work}/runs/{r}.parquet --name {r} --force', f'T8_{r}')
sub = json.load(open(f'{K.work}/submissions/structure_G_rr.json', encoding='utf-8'))
print(len(sub), 'câu hỏi; ví dụ:', json.dumps(sub[0], ensure_ascii=False)[:600])

In [ ]:
# (9) Tổng kết thời gian và dung lượng → gửi kết quả ô (2), (3) và bảng này cho người duyệt
report('e2e')